# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(enrichR)
library(dplyr)
library(openxlsx)
library(ggplot2)
library(readxl)
library(tidyr)
library(viridis)

# Load Data

In [4]:
# List all DEG files in the subdirectories 
files <- list.files(path = "05_DEGs", pattern = "_results.csv$", full.names = TRUE, recursive = TRUE)

# Initialize an empty vector to hold files with significant DEGs
valid_files <- c()

# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file, row.names = 1)
  
  # Extract comparison name 
  comparison <- gsub("_results.csv", "", basename(file))
  
  # Filter to only significant DEGs (adj.P.Val < 0.01)
  significant_df <- df[df$adj.P.Val < 0.01 & !is.na(df$adj.P.Val), ]
  
  # Check the number of significant DEGs
  num_significant <- nrow(significant_df)
  
  print(paste("File:", basename(file), "- Number of significant DEGs:", num_significant))
  
  # If there are significant DEGs, keep the file
  if (num_significant > 0) {
    valid_files <- c(valid_files, file)
  }
}

# Update the original files list to only include valid files
files <- valid_files

[1] "File: MUT_Guide6_vs_MUT_NonEffector_results.csv - Number of significant DEGs: 5123"
[1] "File: MUT_Guide6_vs_MUT_NT_results.csv - Number of significant DEGs: 6346"
[1] "File: MUT_Guide6_vs_WT_NT_results.csv - Number of significant DEGs: 7532"
[1] "File: MUT_NonEffector_vs_MUT_NT_results.csv - Number of significant DEGs: 5540"
[1] "File: MUT_NonEffector_vs_WT_NT_results.csv - Number of significant DEGs: 6813"
[1] "File: MUT_NT_vs_WT_NT_results.csv - Number of significant DEGs: 4334"


# Plot DEGs

## Gene Ontology

In [5]:
# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file)

  # Filter to only significant DEGs
  df <- df[df$adj.P.Val < 0.01, ]

  # Sort by adjusted p-value/logFC and take the top 500
  df <- df[order(df$adj.P.Val), ]
  df <- df[order(abs(df$logFC), decreasing = TRUE), ]
  df <- head(df, 500)
    
  # Perform the enrichR analysis on the gene list
  enrichr_results <- enrichr(df$X, "HDSigDB_Human_2021")
  
  # Create a new Excel workbook for the current file
  wb <- createWorkbook()
  
  # Loop through each table in enrichr_results and save it as a separate sheet in the Excel workbook
  for (i in seq_along(enrichr_results)) {
    # Extract the data frame from the list
    enrichr_df <- enrichr_results[[i]]
    
    # Define the original sheet name
    original_sheet_name <- names(enrichr_results)[i]
    
    # Modify the sheet name if it's too long
    sheet_name <- if (original_sheet_name == "RNA-Seq_Disease_Gene_and_Drug_Signatures_from_GEO") {
      "RNAseq_DiseaseGene_DrugSigs_GEO"
    } else {
      original_sheet_name
    }
    
    # Add the data frame as a new sheet in the Excel workbook
    addWorksheet(wb, sheet_name)
    writeData(wb, sheet = sheet_name, x = enrichr_df)
  }
    
  # Define the output file name
  output_filename <- paste0("06_DEG_GO/", tools::file_path_sans_ext(basename(file)), "_HD_signature_results.xlsx")
  
  # Save the Excel workbook for this file
  saveWorkbook(wb, output_filename, overwrite = TRUE)
  
  # Print message indicating successful save
  cat("Enrichr results saved for:", file, "->", output_filename, "\n")
}

Uploading data to Enrichr... Done.
  Querying HDSigDB_Human_2021... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/MUT_Guide6_vs_MUT_NonEffector_results.csv -> 06_DEG_GO/MUT_Guide6_vs_MUT_NonEffector_results_HD_signature_results.xlsx 
Uploading data to Enrichr... Done.
  Querying HDSigDB_Human_2021... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/MUT_Guide6_vs_MUT_NT_results.csv -> 06_DEG_GO/MUT_Guide6_vs_MUT_NT_results_HD_signature_results.xlsx 
Uploading data to Enrichr... Done.
  Querying HDSigDB_Human_2021... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/MUT_Guide6_vs_WT_NT_results.csv -> 06_DEG_GO/MUT_Guide6_vs_WT_NT_results_HD_signature_results.xlsx 
Uploading data to Enrichr... Done.
  Querying HDSigDB_Human_2021... Done.
Parsing results... Done.
Enrichr results saved for: 05_DEGs/MUT_NonEffector_vs_MUT_NT_results.csv -> 06_DEG_GO/MUT_NonEffector_vs_MUT_NT_results_HD_signature_results.xlsx 
Uploading data to Enrichr... D

## Plot All Genes

In [7]:
# List all Enrichr Excel files
files <- list.files(path = "06_DEG_GO", pattern = "\\_HD_signature_results.xlsx$", full.names = TRUE, recursive = TRUE)

# View
print(files)

[1] "06_DEG_GO/MUT_Guide6_vs_MUT_NonEffector_results_HD_signature_results.xlsx"
[2] "06_DEG_GO/MUT_Guide6_vs_MUT_NT_results_HD_signature_results.xlsx"         
[3] "06_DEG_GO/MUT_Guide6_vs_WT_NT_results_HD_signature_results.xlsx"          
[4] "06_DEG_GO/MUT_NonEffector_vs_MUT_NT_results_HD_signature_results.xlsx"    
[5] "06_DEG_GO/MUT_NonEffector_vs_WT_NT_results_HD_signature_results.xlsx"     
[6] "06_DEG_GO/MUT_NT_vs_WT_NT_results_HD_signature_results.xlsx"              


In [8]:
# Initialize an empty dataframe
all_data <- data.frame()

# Iterate over each file and read in the corresponding Excel sheets
for (file_path in files) {
  # Extract the comparison name from the file name
  comparison <- tools::file_path_sans_ext(basename(file_path))
      
  # Get the names of the sheets in the Excel file
  databases <- excel_sheets(file_path)
  
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file_path, sheet = database)
      
      if (nrow(df) == 0) {
        next
      }
      
      # Select the required columns and add module (comparison) and database information
      df <- df %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = comparison, Database = database)

      # Filter out rows by significance
      df <- df %>%
      filter(Adjusted.P.value <= 0.05)
        
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Rename
all_data$Module <- gsub("_results_HD_signature_results$", "", all_data$Module)

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Top Neuron-Enriched Genes In Humans And Mice PMID29892006,8.937710e-46,6.681818,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021
2,Top Neuron-Specific Genes In Humans And Mice PMID29892006,2.981684e-45,6.600889,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021
3,Genes Changed In HD Vs Normal NPCs GSE118088,2.112452e-32,3.171683,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021
4,Definite Neuron-Enriched Genes PMID18171944,1.138937e-27,8.850003,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021
5,Genes Changed In HD Vs Control Fibroblast-Derived Neurons GSE84013,7.212679e-26,4.555038,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021
6,Probable Neuron-Enriched Genes PMID18171944,1.629809e-24,3.461384,MUT_Guide6_vs_MUT_NonEffector,HDSigDB_Human_2021


In [20]:
# Count terms for human data at different significance cutoffs
cutoffs <- c(0.05, 0.04, 0.03, 0.02, 0.01, 0.005, 0.001,
             0.00001, 0.000001, 0.0000001, 0.00000001, 0.000000001,
            0.0000000001, 0.00000000001, 0.000000000001, 0.0000000000001,
            0.00000000000001, 0.000000000000001, 0.0000000000000001, 0.00000000000000001)

term_counts <- all_data %>%
  crossing(Cutoff = cutoffs) %>%
  filter(!is.na(Adjusted.P.value), Adjusted.P.value <= Cutoff) %>%
  group_by(Database, Cutoff) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term),
    .groups = "drop"
  ) %>%
  arrange(Database, desc(Cutoff))

# View
print(term_counts)

# A tibble: 20 × 4
   Database           Cutoff Total_Terms Unique_Terms
   <chr>               <dbl>       <int>        <int>
 1 HDSigDB_Human_2021  5e- 2        1845          804
 2 HDSigDB_Human_2021  4e- 2        1778          778
 3 HDSigDB_Human_2021  3e- 2        1665          743
 4 HDSigDB_Human_2021  2e- 2        1554          706
 5 HDSigDB_Human_2021  1e- 2        1388          644
 6 HDSigDB_Human_2021  5e- 3        1251          587
 7 HDSigDB_Human_2021  1e- 3        1030          519
 8 HDSigDB_Human_2021  1e- 5         634          342
 9 HDSigDB_Human_2021  1e- 6         508          277
10 HDSigDB_Human_2021  1e- 7         434          245
11 HDSigDB_Human_2021  1e- 8         375          214
12 HDSigDB_Human_2021  1e- 9         320          186
13 HDSigDB_Human_2021  1e-10         279          166
14 HDSigDB_Human_2021  1e-11         235          141
15 HDSigDB_Human_2021  1e-12         193          119
16 HDSigDB_Human_2021  1e-13         173          109
17 HDSigD

In [22]:
# Filter to desired cutoff
filtered_data <- all_data %>%
  filter(Adjusted.P.value <= 1e-14)

# Dot Plot

In [25]:
# Plot
dot_plot <- ggplot(filtered_data, aes(x = gsub("_", " ", Module), y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
  geom_point(shape = 21) +
  scale_fill_viridis() +
  xlab('') + ylab('') +
  labs(title = "Huntington's Disease Signature Pathways") +
  theme(
    panel.background = element_rect(fill = "white", color = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    plot.background = element_rect(fill = "white", color = NA),
    axis.text.x = element_text(angle = 90, hjust = 1),
    axis.text.y = element_text(size = 6)
  )

# Save
ggsave(filename = "00_FIGURES/HD_signatures_dot_plot_1e-14_cutoff.pdf", plot = dot_plot, height = 12, width = 11, limitsize = FALSE)

# Bar Plot

In [38]:
# Significant HD signature terms
module_bar_plot <- ggplot(
  module_counts,
  aes(x = count, y = reorder(gsub("_", " ", Module), count))
) +
  geom_col(fill = "steelblue") +
  geom_text(aes(label = count), hjust = -0.2, size = 2) +
  theme_classic() +
  theme(
    axis.text = element_text(size = 6),
    axis.title = element_text(size = 8),
    plot.title = element_text(size = 10)
  ) +
  labs(
    x = "Number of Significant HD Signature Terms",
    y = "DEG Comparison",
    title = "Significant HD Signature Terms\n(Adj. P-val < 0.05)"
  ) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.1)))

ggsave("00_FIGURES/HD_signature_terms_by_module.pdf", module_bar_plot, width = 5, height = 3)